从 `primekg/knowledge_graph/build_graph.ipynb` 修改而来、取消所有的反向边、删除anatomy和exposure

In [2]:

import numpy as np
import pandas as pd
import igraph as ig

data_path = '../../../data/data/'
save_path = '../kg/'

# Read datasets

In [29]:
# 检查所有数据都是字符串类型
def assert_dtypes(df): 
    all_string = True
    for i, x in enumerate(df.dtypes.values): 
        if x != np.dtype('O'): 
            all_string = False
            print(df.columns[i], x)
    if not all_string: assert False

In [30]:
# df_ppi = pd.read_csv(data_path+'ppi/protein_protein.csv', low_memory=False).dropna()
df_ppi = pd.read_csv(data_path+'ppi/df_ppi_physical.csv', low_memory=False).dropna()
df_ppi = df_ppi.astype({'proteinA_entrezid':int}).astype({'proteinA_entrezid':str})
df_ppi = df_ppi.astype({'proteinB_entrezid':int}).astype({'proteinB_entrezid':str})
assert_dtypes(df_ppi)

In [31]:
df_drugbank = pd.read_csv(data_path+'drugbank/drug_protein.csv', low_memory=False)
df_drugbank = df_drugbank[['DrugBank', 'relation', 'NCBIGeneID','DrugBankName']].dropna()
df_drugbank = df_drugbank.astype({'NCBIGeneID':int}).astype({'NCBIGeneID':str})
assert_dtypes(df_drugbank)

In [32]:
df_disgenet = pd.read_csv(data_path+'disgenet/curated_gene_disease_associations.tsv', sep='\t', low_memory=False)
df_disgenet = df_disgenet.astype({'geneId':int}).astype({'geneId':str})

In [33]:
# df_mondo_terms = pd.read_csv(data_path+'mondo/mondo_terms.csv', low_memory=False)
df_mondo_terms = pd.read_csv(data_path+'mondo/mondo_terms_2.csv', low_memory=False)
df_mondo_terms = df_mondo_terms.astype({'id':int}).astype({'id':str})

df_mondo_xref = pd.read_csv(data_path+'mondo/mondo_references.csv', low_memory=False)
df_mondo_xref = df_mondo_xref.astype({'mondo_id':int}).astype({'mondo_id':str})
assert_dtypes(df_mondo_xref)

# df_mondo_parents = pd.read_csv(data_path+'mondo/mondo_parents.csv', low_memory=False)
df_mondo_parents = pd.read_csv(data_path+'mondo/mondo_parents_fix_new.csv', low_memory=False)
df_mondo_parents = df_mondo_parents.astype({'parent':int}).astype({'parent':str})
df_mondo_parents = df_mondo_parents.astype({'child':int}).astype({'child':str})
assert_dtypes(df_mondo_parents)

In [34]:
df_drug_central = pd.read_csv(data_path+'drugcentral/drug_disease.csv', low_memory=False)
df_drug_central = df_drug_central[['cas_reg_no','relationship_name', 'umls_cui']] # 'concept_id', 'concept_name', 'snomed_conceptid'
df_drug_central = df_drug_central.query('not @df_drug_central.cas_reg_no.isna()')
df_drug_central = df_drug_central.query('not @df_drug_central.umls_cui.isna()')
assert_dtypes(df_drug_central)

In [35]:
df_hp_terms = pd.read_csv(data_path+'hpo/hp_terms.csv', low_memory=False)
df_hp_terms = df_hp_terms.astype({'id':int}).astype({'id':str})

df_hp_xref = pd.read_csv(data_path+'hpo/hp_references.csv', low_memory=False)
df_hp_xref = df_hp_xref.astype({'hp_id':int}).astype({'hp_id':str})

df_hp_parents = pd.read_csv(data_path+'hpo/hp_parents.csv', low_memory=False)
df_hp_parents = df_hp_parents.astype({'parent':int}).astype({'parent':str})
df_hp_parents = df_hp_parents.astype({'child':int}).astype({'child':str})
assert_dtypes(df_hp_parents)

In [36]:
df_ddi = pd.read_csv(data_path+'drugbank/drug_drug.csv', low_memory=False)
assert_dtypes(df_ddi)

In [37]:
df_hpoa_pos = pd.read_csv(data_path+'hpo/disease_phenotype_pos.csv', low_memory=False)
df_hpoa_pos = df_hpoa_pos.astype({'hp_id':int}).astype({'hp_id':str})
df_hpoa_pos = df_hpoa_pos.astype({'disease_ontology_id':int}).astype({'disease_ontology_id':str})
assert_dtypes(df_hpoa_pos)

df_hpoa_neg = pd.read_csv(data_path+'hpo/disease_phenotype_neg.csv', low_memory=False)
df_hpoa_neg = df_hpoa_neg.astype({'hp_id':int}).astype({'hp_id':str})
df_hpoa_neg = df_hpoa_neg.astype({'disease_ontology_id':int}).astype({'disease_ontology_id':str})
assert_dtypes(df_hpoa_neg)

In [38]:
df_sider = pd.read_csv(data_path+'sider/sider.csv', low_memory=False)
assert_dtypes(df_sider)

In [39]:
df_go_terms = pd.read_csv(data_path+'go/go_terms_info.csv', low_memory=False)
df_go_terms = df_go_terms.astype({'go_term_id':int}).astype({'go_term_id':str})
assert_dtypes(df_go_terms)

df_go_edges = pd.read_csv(data_path+'go/go_terms_relations.csv', low_memory=False)
df_go_edges = df_go_edges.astype({'x':int}).astype({'x':str})
df_go_edges = df_go_edges.astype({'y':int}).astype({'y':str})
assert_dtypes(df_go_edges)

df_gene2go = pd.read_csv(data_path+'ncbigene/protein_go_associations.csv', low_memory=False)
df_gene2go = df_gene2go.astype({'ncbi_gene_id':int}).astype({'ncbi_gene_id':str})
df_gene2go = df_gene2go.astype({'go_term_id':int}).astype({'go_term_id':str})
assert_dtypes(df_gene2go)

In [ ]:
df_exposures = pd.read_csv(data_path+'ctd/exposure_data.csv', low_memory=False)
df_exposures = df_exposures[['exposurestressorname', 'exposurestressorid',
                  'exposuremarker', 'exposuremarkerid',
                  'diseasename', 'diseaseid',
                  'phenotypename', 'phenotypeid']]
assert_dtypes(df_exposures)

In [41]:
df_uberon_terms = pd.read_csv(data_path+'uberon/uberon_terms.csv', low_memory=False)
df_uberon_terms = df_uberon_terms.astype({'id':int}).astype({'id':str})
assert_dtypes(df_uberon_terms)

In [42]:
df_uberon_is_a = pd.read_csv(data_path+'uberon/uberon_is_a.csv', low_memory=False)
df_uberon_is_a = df_uberon_is_a.astype({'id':int}).astype({'id':str})
df_uberon_is_a = df_uberon_is_a.astype({'is_a':int}).astype({'is_a':str})
assert_dtypes(df_uberon_is_a)

df_uberon_rels = pd.read_csv(data_path+'uberon/uberon_rels.csv', low_memory=False)
df_uberon_rels = df_uberon_rels.astype({'id':int}).astype({'id':str})
df_uberon_rels = df_uberon_rels.astype({'relation_id':int}).astype({'relation_id':str})
assert_dtypes(df_uberon_rels)

In [43]:
df_bgee = pd.read_csv(data_path+'bgee/anatomy_gene.csv', low_memory=False)
df_bgee = df_bgee.astype({'expression_rank':int}).astype({'expression_rank':str})
df_bgee = df_bgee.astype({'anatomy_id':int}).astype({'anatomy_id':str})
assert_dtypes(df_bgee)

In [44]:
df_reactome_terms = pd.read_csv(data_path+'reactome/reactome_terms.csv', low_memory=False)
assert_dtypes(df_reactome_terms)

df_reactome_rels = pd.read_csv(data_path+'reactome/reactome_relations.csv', low_memory=False)
assert_dtypes(df_reactome_rels)

df_reactome_ncbi = pd.read_csv(data_path+'reactome/reactome_ncbi.csv', low_memory=False)
df_reactome_ncbi = df_reactome_ncbi[df_reactome_ncbi.ncbi_id.str.isnumeric()]
assert_dtypes(df_reactome_ncbi)

In [45]:
df_drug_kegg_pathway = pd.read_csv(data_path+'kegg/drug_kegg_pathway.csv', low_memory=False)
assert_dtypes(df_drug_kegg_pathway)

df_reactome_kegg = pd.read_csv(data_path+'kegg/reactome_kegg.csv', low_memory=False)
assert_dtypes(df_reactome_kegg)

df_drug_reactome_pathway = pd.read_csv(data_path+'kegg/drug_reactome_pathway.csv', low_memory=False)
assert_dtypes(df_drug_reactome_pathway)

额外词汇文件

In [46]:
df_umls_mondo = pd.read_csv(data_path+'vocab/umls_mondo.csv', low_memory=False)
df_umls_mondo = df_umls_mondo.astype({'mondo_id':int}).astype({'mondo_id':str})
assert_dtypes(df_umls_mondo)

In [47]:
df_prot_names = pd.read_csv(data_path+'vocab/gene_names.csv', low_memory=False, sep='\t')
df_prot_names = df_prot_names.rename(columns={'NCBI Gene ID(supplied by NCBI)':'ncbi_id', 'NCBI Gene ID':'ncbi_id2', 'Approved symbol':'symbol', 'Approved name':'name'})
df_prot_names = df_prot_names[['ncbi_id', 'symbol']].dropna()
df_prot_names = df_prot_names.astype({'ncbi_id':int}).astype({'ncbi_id':str})
assert_dtypes(df_prot_names)

In [48]:
db_vocab = pd.read_csv(data_path+'vocab/drugbank_vocabulary.csv', low_memory=False)
assert_dtypes(db_vocab)

In [100]:
df_db_atc = pd.read_csv(data_path+'vocab/drugbank_atc_codes.csv', low_memory=False)[['atc_code','parent_key']]
assert_dtypes(df_db_atc)

# Converting databases into graph edges

In [50]:
def clean_edges(df):
    df = df.get(['relation', 'display_relation', 'x_id','x_type', 'x_name', 'x_source','y_id','y_type', 'y_name', 'y_source']) # select columns
    df = df.dropna()  # 删除空数据
    df = df.drop_duplicates() # 删除重复数据
    df = df.query('not ((x_id == y_id) and (x_type == y_type) and (x_source == y_source) and (x_name == y_name))')  # 删除自环
    return df

In [51]:
# 全部不再添加反向边
# def add_reverse_edges(df):
#     df_rev = df.copy().rename(columns={'x_id':'y_id','x_type':'y_type', 'x_name':'y_name', 'x_source':'y_source',
#                             'y_id':'x_id','y_type':'x_type', 'y_name':'x_name', 'y_source':'x_source' })
#     new_df = pd.concat([df, df_rev], ignore_index=True)
#     return new_df

## Basic

### 附加：df_drug_path (KEGG)

In [66]:
df_drug_path1 = pd.merge(df_drug_kegg_pathway, db_vocab, 'left', left_on='drugbank_id', right_on='DrugBank ID')

df_drug_path1 = df_drug_path1[['drugbank_id', 'Common name', 'pathway_kegg_id', 'pathway_kegg_name']]
df_drug_path1 = df_drug_path1.dropna().drop_duplicates()

df_drug_path1 = df_drug_path1.rename(columns={'drugbank_id':'x_id', 'Common name':'x_name', 'pathway_kegg_id':'y_id', 'pathway_kegg_name':'y_name'})
df_drug_path1['x_type'] = 'drug'
df_drug_path1['x_source'] = 'DrugBank'
df_drug_path1['y_type'] = 'pathway'
df_drug_path1['y_source'] = 'KEGG'
df_drug_path1['relation'] = 'drug_pathway'
df_drug_path1['display_relation'] = 'drug_pathway'
df_drug_path1 = clean_edges(df_drug_path1)
print("KEGG 引入的 drug pathway 关系数量:", df_drug_path1.shape[0])
print("KEGG 提到的 drug 数量:", df_drug_path1['x_id'].unique().shape[0])
print("KEGG 提到的 pathway 数量:", df_drug_path1['y_id'].unique().shape[0])
print("KEGG 中 drug 的平均出度数:", df_drug_path1.groupby('x_id').size().mean())
print("KEGG 中 pathway 的平均入度数:", df_drug_path1.groupby('y_id').size().mean())
df_drug_path1.head(1)

KEGG 引入的 drug pathway 关系数量: 2595
KEGG 提到的 drug 数量: 1640
KEGG 提到的 pathway 数量: 106
KEGG 中 drug 的平均出度数: 1.5823170731707317
KEGG 中 pathway 的平均入度数: 24.4811320754717


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,drug_pathway,drug_pathway,DB13083,drug,Talarozole,DrugBank,hsa00830,pathway,Retinol metabol,KEGG


### df_drug_path (REACTOME)

In [67]:
df_drug_path2 = pd.merge(df_drug_reactome_pathway, db_vocab, 'left', left_on='drugbank_id', right_on='DrugBank ID')
df_drug_path2 = df_drug_path2[['drugbank_id', 'Common name', 'pathway_reactome_id', 'pathway_reactome_name']]
df_drug_path2 = df_drug_path2.dropna().drop_duplicates()

df_drug_path2 = df_drug_path2.rename(columns={'drugbank_id':'x_id', 'Common name':'x_name', 'pathway_reactome_id':'y_id', 'pathway_reactome_name':'y_name'})
df_drug_path2['x_type'] = 'drug'
df_drug_path2['x_source'] = 'DrugBank'
df_drug_path2['y_type'] = 'pathway'
df_drug_path2['y_source'] = 'REACTOME'
df_drug_path2['relation'] = 'drug_pathway'
df_drug_path2['display_relation'] = 'drug_pathway'
df_drug_path2 = clean_edges(df_drug_path2)
print("REACTOME 引入的 drug pathway 关系数量:", df_drug_path2.shape[0])
print("REACTOME 提到的 drug 数量:", df_drug_path2['x_id'].unique().shape[0])
print("REACTOME 提到的 pathway 数量:", df_drug_path2['y_id'].unique().shape[0])
print("REACTOME 中 drug 的平均出度数:", df_drug_path2.groupby('x_id').size().mean())
print("REACTOME 中 pathway 的平均入度数:", df_drug_path2.groupby('y_id').size().mean())
df_drug_path2.head(1)

REACTOME 引入的 drug pathway 关系数量: 657
REACTOME 提到的 drug 数量: 523
REACTOME 提到的 pathway 数量: 41
REACTOME 中 drug 的平均出度数: 1.2562141491395793
REACTOME 中 pathway 的平均入度数: 16.024390243902438


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,drug_pathway,drug_pathway,DB01119,drug,Diazoxide,DrugBank,R-HSA-422356,pathway,Regulation of insulin secretion,REACTOME


In [70]:
df_drug_path = pd.concat([df_drug_path1, df_drug_path2], ignore_index=True)
# df_drug_path = add_reverse_edges(df_drug_path)
print("drug pathway 总关系数量:", df_drug_path.shape[0])
print("drug pathway 总 drug 数量:", df_drug_path['x_id'].unique().shape[0])
print("drug pathway 总 pathway 数量:", df_drug_path['y_id'].unique().shape[0])
print("drug pathway 中 drug 的平均出度数:", df_drug_path.groupby('x_id').size().mean())
print("drug pathway 中 pathway 的平均入度数:", df_drug_path.groupby('y_id').size().mean())
df_drug_path.head(1)

drug pathway 总关系数量: 3252
drug pathway 总 drug 数量: 1868
drug pathway 总 pathway 数量: 147
drug pathway 中 drug 的平均出度数: 1.740899357601713
drug pathway 中 pathway 的平均入度数: 22.122448979591837


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,drug_pathway,drug_pathway,DB13083,drug,Talarozole,DrugBank,hsa00830,pathway,Retinol metabol,KEGG


### REACTOME KEGG 关系

In [72]:
df_path_path_add = df_reactome_kegg.copy()
df_path_path_add = clean_edges(df_path_path_add)
print("pathway pathway 关系数量:", df_path_path_add.shape[0])
df_path_path_add.head(1)

pathway pathway 关系数量: 106


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,pathway_pathway,parent-child,R-HSA-211916,pathway,Vitamins,REACTOME,hsa00830,pathway,Retinol metabol,KEGG


### Protein protein interactions (NCBI) PPI

In [74]:
# 补充：首先单向关系
df_ppi['pair'] = df_ppi.apply(lambda row: tuple(sorted([row['proteinA_entrezid'], row['proteinB_entrezid']])), axis=1)

df_ppi = df_ppi.drop_duplicates('pair', keep='first')
df_ppi = df_ppi.drop('pair', axis=1)

df_prot_prot = pd.merge(df_ppi, df_prot_names, 'left', left_on='proteinA_entrezid', right_on='ncbi_id').rename(columns={'symbol':'symbolA'})
df_prot_prot = pd.merge(df_ppi, df_prot_names, 'left', left_on='proteinA_entrezid', right_on='ncbi_id').drop(columns=['symbolA']).rename(columns={'symbol':'symbolA'})  

df_prot_prot = pd.merge(df_prot_prot, df_prot_names, 'left', left_on='proteinB_entrezid', right_on='ncbi_id').rename(columns={'symbol':'symbolB'})
df_prot_prot = pd.merge(df_prot_prot, df_prot_names, 'left', left_on='proteinB_entrezid', right_on='ncbi_id').drop(columns=['symbolB']).rename(columns={'symbol':'symbolB'})

df_prot_prot = df_prot_prot.rename(columns={'proteinA_entrezid':'x_id', 'proteinB_entrezid':'y_id', 'symbolA':'x_name', 'symbolB':'y_name'})
df_prot_prot['x_type'] = 'gene/protein'
df_prot_prot['x_source'] = 'NCBI'
df_prot_prot['y_type'] = 'gene/protein'
df_prot_prot['y_source'] = 'NCBI'
df_prot_prot['relation'] = 'protein_protein'  # display_relation and relation 等价
df_prot_prot['display_relation'] = 'ppi'
# df_prot_prot = add_reverse_edges(df_prot_prot)
df_prot_prot = clean_edges(df_prot_prot)
print("protein protein 关系数量:", df_prot_prot.shape[0])
print("protein protein 中 protein 数量:", pd.concat([df_prot_prot['x_id'], df_prot_prot['y_id']]).unique().shape[0])
print("protein protein 中 protein 的平均出度数:", df_prot_prot.groupby('x_id').size().mean())
print("protein protein 中 protein 的平均入度数:", df_prot_prot.groupby('y_id').size().mean())
df_prot_prot.head(1)

protein protein 关系数量: 715533
protein protein 中 protein 数量: 18206
protein protein 中 protein 的平均出度数: 42.61915539936864
protein protein 中 protein 的平均入度数: 41.02356381148951


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,protein_protein,ppi,381,gene/protein,ARF5,NCBI,4907,gene/protein,NT5E,NCBI


### Drug protein interactions (DrugBank)

In [76]:
df_prot_drug = pd.merge(df_drugbank, df_prot_names, 'left', left_on='NCBIGeneID', right_on='ncbi_id')

df_prot_drug = df_prot_drug.rename(columns={'DrugBank':'x_id', 'NCBIGeneID':'y_id', 'DrugBankName':'x_name', 'symbol':'y_name'})
df_prot_drug['x_type'] = 'drug'
df_prot_drug['x_source'] = 'DrugBank'
df_prot_drug['y_type'] = 'gene/protein'
df_prot_drug['y_source'] = 'NCBI'
df_prot_drug['display_relation'] = df_prot_drug['relation'].values
# print(df_prot_drug['display_relation'].unique())
df_prot_drug['relation'] = 'drug_protein' # combine targets, carrier, enzyme and transporter
# df_prot_drug = add_reverse_edges(df_prot_drug)
df_prot_drug = clean_edges(df_prot_drug)
print("drug protein 关系数量:", df_prot_drug.shape[0])
print("drug protein 中 drug 数量:", df_prot_drug['x_id'].unique().shape[0])
print("drug protein 中 protein 数量:", df_prot_drug['y_id'].unique().shape[0])
print("drug protein 中 drug 的平均出度数:", df_prot_drug.groupby('x_id').size().mean())
print("drug protein 中 protein 的平均入度数:", df_prot_drug.groupby('y_id').size().mean())
df_prot_drug.head(1)

drug protein 关系数量: 31476
drug protein 中 drug 数量: 7893
drug protein 中 protein 数量: 3309
drug protein 中 drug 的平均出度数: 3.9878373242113265
drug protein 中 protein 的平均入度数: 9.512239347234814


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,drug_protein,carrier,DB09130,drug,Copper,DrugBank,2157,gene/protein,F8,NCBI


### Drug disease interactions (DiseaseCentral) –– PENDING

In [78]:
df_drug_dis = pd.merge(df_drug_central, db_vocab, 'left', left_on='cas_reg_no', right_on='CAS')
df_drug_dis = pd.merge(df_drug_dis, df_umls_mondo, 'inner', left_on='umls_cui', right_on='umls_id')
df_drug_dis = pd.merge(df_drug_dis, df_mondo_terms, 'left', left_on='mondo_id', right_on='id')

df_drug_dis = df_drug_dis[['relationship_name','DrugBank ID', 'Common name', 'mondo_id', 'name']]
df_drug_dis = df_drug_dis.dropna().drop_duplicates()

df_drug_dis = df_drug_dis.rename(columns={'DrugBank ID':'x_id', 'mondo_id':'y_id', 'Common name':'x_name', 'name':'y_name', 'relationship_name':'relation'})
df_drug_dis['x_type'] = 'drug'
df_drug_dis['x_source'] = 'DrugBank'
df_drug_dis['y_type'] = 'disease'
df_drug_dis['y_source'] = 'MONDO'
df_drug_dis['display_relation'] = df_drug_dis['relation'].values
# print(df_drug_dis['display_relation'].unique())
# df_drug_dis = add_reverse_edges(df_drug_dis)
print("drug disease 关系数量:", df_drug_dis.shape[0])
print("drug disease 中 drug 数量:", df_drug_dis['x_id'].unique().shape[0])
print("drug disease 中 disease 数量:", df_drug_dis['y_id'].unique().shape[0])
print("drug disease 中 drug 的平均出度数:", df_drug_dis.groupby('x_id').size().mean())
print("drug disease 中 disease 的平均入度数:", df_drug_dis.groupby('y_id').size().mean())
df_drug_dis.head(1)

drug disease 关系数量: 28861
drug disease 中 drug 数量: 2261
drug disease 中 disease 数量: 1707
drug disease 中 drug 的平均出度数: 12.764705882352942
drug disease 中 disease 的平均入度数: 16.907439953134155


,relation,x_id,x_name,y_id,y_name,x_type,x_source,y_type,y_source,display_relation
0,contraindication,DB00435,Nitric Oxide,1117,methemoglobinemia,drug,DrugBank,disease,MONDO,contraindication


### Disease protein interactions (DisGenNet)

In [79]:

# df_prot_dis1 = df_disgenet.query('diseaseType=="disease"')
# 缺失 diseaseType 列
df_prot_dis1 = df_disgenet.copy()

df_prot_dis1 = pd.merge(df_prot_dis1, df_umls_mondo, 'inner', left_on='diseaseId', right_on='umls_id')
df_prot_dis1 = pd.merge(df_prot_dis1, df_mondo_terms, 'left', left_on='mondo_id', right_on='id')

df_prot_dis1 = df_prot_dis1.rename(columns={'geneId':'x_id', 'geneSymbol':'x_name', 'mondo_id':'y_id', 'name':'y_name'})
df_prot_dis1['x_type'] = 'gene/protein'
df_prot_dis1['x_source'] = 'NCBI'
df_prot_dis1['y_type'] = 'disease'
df_prot_dis1['y_source'] = 'MONDO'
df_prot_dis1['relation'] = 'disease_protein'
df_prot_dis1['display_relation'] = 'associated with'
df_prot_dis1 = clean_edges(df_prot_dis1)
print("disease protein 关系数量:", df_prot_dis1.shape[0])
print("disease protein 中 protein 数量:", df_prot_dis1['x_id'].unique().shape[0])
print("disease protein 中 disease 数量:", df_prot_dis1['y_id'].unique().shape[0])
print("disease protein 中 protein 的平均出度数:", df_prot_dis1.groupby('x_id').size().mean())
print("disease protein 中 disease 的平均入度数:", df_prot_dis1.groupby('y_id').size().mean())
df_prot_dis1.head(1)

disease protein 关系数量: 67339
disease protein 中 protein 数量: 8408
disease protein 中 disease 数量: 7543
disease protein 中 protein 的平均出度数: 8.008920076117983
disease protein 中 disease 的平均入度数: 8.92734986079809


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,disease_protein,associated with,10,gene/protein,NAT2,NCBI,4987,disease,urinary bladder neoplasm,MONDO


### Disease disease interations (MONDO)

In [80]:
df_dis_dis1 = pd.merge(df_mondo_parents, df_mondo_terms, 'left', left_on='parent', right_on='id')
df_dis_dis1 = df_dis_dis1.rename(columns={'parent':'x_id', 'name':'x_name'})
df_dis_dis1 = pd.merge(df_dis_dis1, df_mondo_terms, 'left', left_on='child', right_on='id')
df_dis_dis1 = df_dis_dis1.rename(columns={'child':'y_id', 'name':'y_name'})
df_dis_dis1['x_type'] = 'disease'
df_dis_dis1['x_source'] = 'MONDO'
df_dis_dis1['y_type'] = 'disease'
df_dis_dis1['y_source'] = 'MONDO'
df_dis_dis1['relation'] = 'disease_disease'
df_dis_dis1['display_relation'] = 'parent-child'
df_dis_dis1 = clean_edges(df_dis_dis1)
print("disease disease 关系数量:", df_dis_dis1.shape[0])
print("disease disease 中 disease 数量:", pd.concat([df_dis_dis1['x_id'], df_dis_dis1['y_id']]).unique().shape[0])
print("disease disease 中 disease 的平均出度数:", df_dis_dis1.groupby('x_id').size().mean())
print("disease disease 中 disease 的平均入度数:", df_dis_dis1.groupby('y_id').size().mean())
df_dis_dis1.head(1)

disease disease 关系数量: 67006
disease disease 中 disease 数量: 27957
disease disease 中 disease 的平均出度数: 6.1580737064608035
disease disease 中 disease 的平均入度数: 2.3988114416639816


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,disease_disease,parent-child,1,disease,disease,MONDO,2,disease,"obsolete 46,XX sex reversal",MONDO


### Drug drug interactions (DrugBank)

In [81]:
df_drug_drug = pd.merge(df_ddi, db_vocab, 'left', left_on='drug1', right_on='DrugBank ID')
df_drug_drug = df_drug_drug.rename(columns={'drug1':'x_id', 'Common name':'x_name'})
df_drug_drug = pd.merge(df_drug_drug.astype({'drug2':'str'}), db_vocab, 'left', left_on='drug2', right_on='DrugBank ID')
df_drug_drug = df_drug_drug.rename(columns={'drug2':'y_id', 'Common name':'y_name'})
df_drug_drug['x_type'] = 'drug'
df_drug_drug['x_source'] = 'DrugBank'
df_drug_drug['y_type'] = 'drug'
df_drug_drug['y_source'] = 'DrugBank'
df_drug_drug['relation'] = 'drug_drug'
df_drug_drug['display_relation'] = 'synergistic interaction'
# df_drug_drug = add_reverse_edges(df_drug_drug)
df_drug_drug = clean_edges(df_drug_drug)
print("drug drug 关系数量:", df_drug_drug.shape[0])
print("drug drug 中 drug 数量:", pd.concat([df_drug_drug['x_id'], df_drug_drug['y_id']]).unique().shape[0])
print("drug drug 中 drug 的平均出度数:", df_drug_drug.groupby('x_id').size().mean())
print("drug drug 中 drug 的平均入度数:", df_drug_drug.groupby('y_id').size().mean())
print(df_drug_drug.shape)
df_drug_drug.head(1)

drug drug 关系数量: 2855310
drug drug 中 drug 数量: 4566
drug drug 中 drug 的平均出度数: 625.3416557161629
drug drug 中 drug 的平均入度数: 625.3416557161629
(2855310, 10)


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,drug_drug,synergistic interaction,DB00001,drug,Lepirudin,DrugBank,DB06605,drug,Apixaban,DrugBank


## Effect/Phenotype

### Effect protein interactions (DisGenNet)

In [83]:

# df_prot_phe = df_disgenet.query('diseaseType=="phenotype"')
# 缺失 diseaseType 列
df_prot_phe = df_disgenet.copy()

df_prot_phe = pd.merge(df_prot_phe, df_hp_xref, 'inner', left_on='diseaseId', right_on='ontology_id')
df_prot_phe = pd.merge(df_prot_phe, df_hp_terms, 'left', left_on='hp_id', right_on='id')

df_prot_phe = df_prot_phe.rename(columns={'geneId':'x_id', 'geneSymbol':'x_name', 'hp_id':'y_id', 'name':'y_name'})
df_prot_phe['x_type'] = 'gene/protein'
df_prot_phe['x_source'] = 'NCBI'
df_prot_phe['y_type'] = 'effect/phenotype'
df_prot_phe['y_source'] = 'HPO'
df_prot_phe['relation'] = 'phenotype_protein'
df_prot_phe['display_relation'] = 'associated with'
# df_prot_phe = add_reverse_edges(df_prot_phe)
df_prot_phe = clean_edges(df_prot_phe)
print("phenotype protein 关系数量:", df_prot_phe.shape[0])
print("phenotype protein 中 protein 数量:", df_prot_phe['x_id'].unique().shape[0])
print("phenotype protein 中 phenotype 数量:", df_prot_phe['y_id'].unique().shape[0])
print("phenotype protein 中 protein 的平均出度数:", df_prot_phe.groupby('x_id').size().mean())
print("phenotype protein 中 phenotype 的平均入度数:", df_prot_phe.groupby('y_id').size().mean())
df_prot_phe.head(1)

phenotype protein 关系数量: 85132
phenotype protein 中 protein 数量: 6541
phenotype protein 中 phenotype 数量: 5819
phenotype protein 中 protein 的平均出度数: 13.015135300412782
phenotype protein 中 phenotype 的平均入度数: 14.630005155525005


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,phenotype_protein,associated with,10,gene/protein,NAT2,NCBI,9725,effect/phenotype,Bladder neoplasm,HPO


### Effect effect interactions (HPO)

In [90]:
df_phe_phe = pd.merge(df_hp_parents, df_hp_terms, 'left', left_on='parent', right_on='id')
df_phe_phe = df_phe_phe.rename(columns={'name':'parent_name'})
df_phe_phe = pd.merge(df_phe_phe, df_hp_terms, 'left', left_on='child', right_on='id')
df_phe_phe = df_phe_phe.rename(columns={'name':'child_name'})
df_phe_phe = df_phe_phe[['parent', 'child', 'parent_name', 'child_name']]

df_phe_phe = df_phe_phe.rename(columns={'parent':'x_id', 'child':'y_id', 'parent_name':'x_name', 'child_name':'y_name'})
df_phe_phe['x_type'] = 'effect/phenotype'
df_phe_phe['x_source'] = 'HPO'
df_phe_phe['y_type'] = 'effect/phenotype'
df_phe_phe['y_source'] = 'HPO'
df_phe_phe['relation'] = 'phenotype_phenotype'
df_phe_phe['display_relation'] = 'parent-child'
df_phe_phe = clean_edges(df_phe_phe)
print("phenotype phenotype 关系数量:", df_phe_phe.shape[0])
print("phenotype phenotype 中 phenotype 数量:", pd.concat([df_phe_phe['x_id'], df_phe_phe['y_id']]).unique().shape[0])
print("phenotype phenotype 中 phenotype 的平均出度数:", df_phe_phe.groupby('x_id').size().mean())
print("phenotype phenotype 中 phenotype 的平均入度数:", df_phe_phe.groupby('y_id').size().mean())
df_phe_phe.head(1)

phenotype phenotype 关系数量: 23528
phenotype phenotype 中 phenotype 数量: 19177
phenotype phenotype 中 phenotype 的平均出度数: 4.029457098818291
phenotype phenotype 中 phenotype 的平均入度数: 1.2269503546099292


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,phenotype_phenotype,parent-child,1507,effect/phenotype,Growth abnormality,HPO,2,effect/phenotype,Abnormality of body height,HPO


### Disease effect interactions (HPO-A)

In [85]:
df_dis_phe_pos1 = pd.merge(df_hpoa_pos, df_mondo_xref, 'left', left_on='disease_ontology_id', right_on='ontology_id')
df_dis_phe_pos1 = df_dis_phe_pos1.query('(disease_ontology==ontology) or (disease_ontology=="ORPHA" and ontology=="Orphanet")')
# 没有 disease_ontology == ontology 的行
df_dis_phe_pos1 = pd.merge(df_dis_phe_pos1, df_hp_terms, 'left', left_on='hp_id', right_on='id').rename(columns={'name':'hp_name'})
df_dis_phe_pos1 = pd.merge(df_dis_phe_pos1, df_mondo_terms, 'left', left_on='mondo_id', right_on='id').rename(columns={'name':'mondo_name'})
df_dis_phe_pos1 = df_dis_phe_pos1[['mondo_id', 'mondo_name', 'hp_id', 'hp_name']]
df_dis_phe_pos1 = df_dis_phe_pos1.rename(columns={'mondo_id':'x_id', 'mondo_name':'x_name', 'hp_id': 'y_id', 'hp_name':'y_name'})
df_dis_phe_pos1.loc[:, 'x_source'] = 'MONDO'
df_dis_phe_pos1.loc[:, 'x_type'] = 'disease'
df_dis_phe_pos1.loc[:, 'y_source'] = 'HPO'
df_dis_phe_pos1.loc[:, 'y_type'] = 'effect/phenotype'
df_dis_phe_pos1.loc[:, 'relation'] = 'disease_phenotype_positive'
df_dis_phe_pos1.loc[:, 'display_relation'] = 'phenotype present'
df_dis_phe_pos1 = clean_edges(df_dis_phe_pos1)
print("disease phenotype positive 关系数量:", df_dis_phe_pos1.shape[0])
print("disease phenotype positive 中 disease 数量:", df_dis_phe_pos1['x_id'].unique().shape[0])
print("disease phenotype positive 中 phenotype 数量:", df_dis_phe_pos1['y_id'].unique().shape[0])
print("disease phenotype positive 中 disease 的平均出度数:", df_dis_phe_pos1.groupby('x_id').size().mean())
print("disease phenotype positive 中 phenotype 的平均入度数:", df_dis_phe_pos1.groupby('y_id').size().mean())
df_dis_phe_pos1.head(1)

disease phenotype positive 关系数量: 251700
disease phenotype positive 中 disease 数量: 10485
disease phenotype positive 中 phenotype 数量: 11350
disease phenotype positive 中 disease 的平均出度数: 24.005722460658085
disease phenotype positive 中 phenotype 的平均入度数: 22.176211453744493


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,disease_phenotype_positive,phenotype present,23659,disease,developmental and epileptic encephalopathy 96,MONDO,11097,effect/phenotype,Epileptic spasm,HPO


In [86]:
df_dis_phe_neg = pd.merge(df_hpoa_neg, df_mondo_xref, 'left', left_on='disease_ontology_id', right_on='ontology_id')
df_dis_phe_neg = df_dis_phe_neg.query('(disease_ontology==ontology) or (disease_ontology=="ORPHA" and ontology=="Orphanet")')
df_dis_phe_neg = pd.merge(df_dis_phe_neg, df_hp_terms, 'left', left_on='hp_id', right_on='id').rename(columns={'name':'hp_name'})
df_dis_phe_neg = pd.merge(df_dis_phe_neg, df_mondo_terms, 'left', left_on='mondo_id', right_on='id').rename(columns={'name':'mondo_name'})
df_dis_phe_neg = df_dis_phe_neg[['mondo_id', 'mondo_name', 'hp_id', 'hp_name']]
df_dis_phe_neg = df_dis_phe_neg.rename(columns={'mondo_id':'x_id', 'mondo_name':'x_name', 'hp_id': 'y_id', 'hp_name':'y_name'})
df_dis_phe_neg.loc[:, 'x_source'] = 'MONDO'
df_dis_phe_neg.loc[:, 'x_type'] = 'disease'
df_dis_phe_neg.loc[:, 'y_source'] = 'HPO'
df_dis_phe_neg.loc[:, 'y_type'] = 'effect/phenotype'
df_dis_phe_neg.loc[:, 'relation'] = 'disease_phenotype_negative'
df_dis_phe_neg.loc[:, 'display_relation'] = 'phenotype absent'
# df_dis_phe_neg = add_reverse_edges(df_dis_phe_neg)
df_dis_phe_neg = clean_edges(df_dis_phe_neg)
print("disease phenotype negative 关系数量:", df_dis_phe_neg.shape[0])
print("disease phenotype negative 中 disease 数量:", df_dis_phe_neg['x_id'].unique().shape[0])
print("disease phenotype negative 中 phenotype 数量:", df_dis_phe_neg['y_id'].unique().shape[0])
print("disease phenotype negative 中 disease 的平均出度数:", df_dis_phe_neg.groupby('x_id').size().mean())
print("disease phenotype negative 中 phenotype 的平均入度数:", df_dis_phe_neg.groupby('y_id').size().mean())
df_dis_phe_neg.head(1)

disease phenotype negative 关系数量: 711
disease phenotype negative 中 disease 数量: 342
disease phenotype negative 中 phenotype 数量: 445
disease phenotype negative 中 disease 的平均出度数: 2.0789473684210527
disease phenotype negative 中 phenotype 的平均入度数: 1.597752808988764


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,disease_phenotype_negative,phenotype absent,16045,disease,tetragametic chimerism,MONDO,1263,effect/phenotype,Global developmental delay,HPO


### Remove phenotype nodes if they exist in MONDO

In [ ]:
# phenotypes that are actually diseases in MONDO
# avoid duplicate nodes and convert them to disease relations
# MONDO 中能映射到 HPO 的
mondo_xref_hp_subset = df_mondo_xref.query('ontology=="HP"')
mondo_xref_hp_subset.loc[:, 'ontology_id'] = mondo_xref_hp_subset['ontology_id'].astype(int).astype(str).values  # 将数据类型转换为 str (HPO 的 id)
# 确保 这些 HPO 的 id 在 HPO 中是有效的
# HPO 中的 id 且是 MONDO 中的疾病类型
hp_ids_r_mondo = pd.merge(mondo_xref_hp_subset, df_hp_terms, 'inner', left_on='ontology_id', right_on='id')['ontology_id'].values

def replace_hp_data_w_mondo(df, hp_id_col, drop_cols=[]): 
    cols = list(df.columns.values)
    cols.extend(['mondo_id', 'mondo_name'])
    [cols.remove(x) for x in drop_cols]
    df = pd.merge(df, mondo_xref_hp_subset, 'left', left_on=hp_id_col, right_on='ontology_id')
    df = pd.merge(df, df_mondo_terms, 'left', left_on='mondo_id', right_on='id')
    df = df.rename(columns={'name':'mondo_name'})[cols]
    return df

In [91]:
# HANDLE phenotype EFFECT 

# PHE-PHE should be PHE-DIS if ONE PHE is in MONDO

df_dis_phe_x = df_phe_phe.query('x_id in @hp_ids_r_mondo and y_id not in @hp_ids_r_mondo')  # x 是一个 MONDO 但 y 不是
df_dis_phe_x = replace_hp_data_w_mondo(df=df_dis_phe_x, hp_id_col='x_id', 
                                       drop_cols=[c for c in df_dis_phe_x.columns.values if 'x_' in c])
df_dis_phe_x = df_dis_phe_x.rename(columns={'mondo_id':'x_id', 'mondo_name':'x_name'})
df_dis_phe_x.loc[:, 'x_source'] = 'MONDO'
df_dis_phe_x.loc[:, 'x_type'] = 'disease'

df_dis_phe_y = df_phe_phe.query('y_id in @hp_ids_r_mondo and x_id not in @hp_ids_r_mondo')
df_dis_phe_y = replace_hp_data_w_mondo(df=df_dis_phe_y, hp_id_col='y_id',
                                       drop_cols=[c for c in df_dis_phe_y.columns.values if 'y_' in c])
df_dis_phe_y = df_dis_phe_y.rename(columns={'mondo_id':'y_id', 'mondo_name':'y_name'})
df_dis_phe_y.loc[:, 'y_source'] = 'MONDO'
df_dis_phe_y.loc[:, 'y_type'] = 'disease'

# 疾病和表型关系

df_dis_phe_pos2 = pd.concat([df_dis_phe_x, df_dis_phe_y], ignore_index=True)
df_dis_phe_pos2['relation'] = 'disease_phenotype_positive'
df_dis_phe_pos2.loc[:, 'display_relation'] = 'phenotype present'
df_dis_phe_pos2 = clean_edges(df_dis_phe_pos2)


# PHE-PHE should be DIS-DIS if BOTH PHE are in MONDO

df_dis_dis2 = df_phe_phe.query('x_id in @hp_ids_r_mondo and y_id in @hp_ids_r_mondo')

df_dis_dis2 = replace_hp_data_w_mondo(df=df_dis_dis2, 
                                       hp_id_col='x_id', 
                                       drop_cols=[c for c in df_dis_dis2.columns.values if 'x_' in c])
df_dis_dis2 = df_dis_dis2.rename(columns={'mondo_id':'x_id', 'mondo_name':'x_name'})
df_dis_dis2 = replace_hp_data_w_mondo(df=df_dis_dis2, 
                                       hp_id_col='y_id', 
                                       drop_cols=[c for c in df_dis_dis2.columns.values if 'y_' in c])
# 疾病和疾病关系

df_dis_dis2 = df_dis_dis2.rename(columns={'mondo_id':'y_id', 'mondo_name':'y_name'})
df_dis_dis2.loc[:, 'x_source'] = 'MONDO'
df_dis_dis2.loc[:, 'x_type'] = 'disease'
df_dis_dis2.loc[:, 'y_source'] = 'MONDO'
df_dis_dis2.loc[:, 'y_type'] = 'disease'
df_dis_dis2.loc[:,'relation'] = 'disease_disease'
df_dis_dis2.loc[:,'display_relation'] = 'parent-child'
df_dis_dis2 = clean_edges(df_dis_dis2)

# drop relations in PHE PHE if either PHE is in MONDO
# phenotype phenotype should have no disease nodes
# 删除原有的 php_phe 中 任何一个实体是 MONDO 的
df_phe_phe = df_phe_phe.query('x_id not in @hp_ids_r_mondo and y_id not in @hp_ids_r_mondo')
print("phenotype-phenotype 关系数量:", df_phe_phe.shape[0])

phenotype-phenotype 关系数量: 22097


In [92]:
# HANDLE protein EFFECT 

# if phenotype in MONDO make it protein-disease relations 
df_prot_dis2= df_prot_phe.query('y_id in @hp_ids_r_mondo')
df_prot_dis2 = replace_hp_data_w_mondo(df=df_prot_dis2, hp_id_col='y_id',
                                       drop_cols=[c for c in df_prot_dis2.columns.values if 'y_' in c])
df_prot_dis2 = df_prot_dis2.rename(columns={'mondo_id':'y_id', 'mondo_name':'y_name'})
df_prot_dis2.loc[:, 'y_source'] = 'MONDO'
df_prot_dis2.loc[:, 'y_type'] = 'disease'
df_prot_dis2.loc[:, 'relation'] = 'disease_protein'
df_prot_dis2.loc[:, 'display_relation'] = 'associated with'
df_prot_dis2 = clean_edges(df_prot_dis2)

# remove from protein-phenotype if phenotype in MONDO 
df_prot_phe = df_prot_phe.query('y_id not in @hp_ids_r_mondo')
print("protein phenotype 关系数量:", df_prot_phe.shape[0])

protein phenotype 关系数量: 72403


In [95]:
# HANDLE disease EFFECT 

# remove from protein-phenotype if phenotype in MONDO 
df_dis_phe_pos1 = df_dis_phe_pos1.query('y_id not in @hp_ids_r_mondo')

# NEGATIVE disease_phenotype should just be dropped because negative disease_disease doesn't make sense 
df_dis_phe_neg = df_dis_phe_neg.query('y_id not in @hp_ids_r_mondo')
print("disease phenotype negative 关系数量:", df_dis_phe_neg.shape[0])

disease phenotype negative 关系数量: 620


In [96]:
# COMBINE DATAFRAMES 

df_prot_dis = pd.concat([df_prot_dis1, df_prot_dis2], ignore_index=True).drop_duplicates()
df_dis_dis = pd.concat([df_dis_dis1, df_dis_dis2], ignore_index=True).drop_duplicates()
df_dis_phe_pos = pd.concat([df_dis_phe_pos1, df_dis_phe_pos2], ignore_index=True).drop_duplicates()

disease phenotype positive 关系数量: 224016


In [97]:
# df_prot_dis = add_reverse_edges(df_prot_dis)
df_prot_dis = clean_edges(df_prot_dis)
print("protein disease 关系数量:", df_prot_dis.shape[0])

protein disease 关系数量: 68749


In [98]:
# df_dis_phe_pos = add_reverse_edges(df_dis_phe_pos)
df_dis_phe_pos = clean_edges(df_dis_phe_pos)
print("disease phenotype positive 关系数量:", df_dis_phe_pos.shape[0])

disease phenotype positive 关系数量: 224016


In [99]:
# df_dis_dis = add_reverse_edges(df_dis_dis)
df_dis_dis = clean_edges(df_dis_dis)
print("disease disease 关系数量:", df_dis_dis.shape[0])

disease disease 关系数量: 67040


### Drug effect interactions (SIDER)

In [101]:
df_drug_effect = pd.merge(df_sider, df_db_atc, 'left', left_on='atc', right_on='atc_code')  # 药物 ATC 编码 映射到 Drugbank
df_drug_effect = df_drug_effect.rename(columns={'parent_key':'DrugBank', 'UMLS_from_meddra':'UMLS'})  # UMLS 映射到 HPO
df_drug_effect = pd.merge(df_drug_effect, db_vocab, 'left', left_on='DrugBank', right_on='DrugBank ID')
df_drug_effect = pd.merge(df_drug_effect, df_hp_xref, 'left', left_on='UMLS' , right_on='ontology_id')
df_drug_effect = pd.merge(df_drug_effect, df_hp_terms, 'left', left_on='hp_id' , right_on='id')
df_drug_effect = df_drug_effect[['DrugBank ID','Common name','hp_id', 'name']]
df_drug_effect = df_drug_effect.dropna().drop_duplicates()

df_drug_effect = df_drug_effect.rename(columns={'DrugBank ID':'x_id', 'Common name':'x_name', 'hp_id':'y_id', 'name':'y_name'})
df_drug_effect['x_type'] = 'drug'
df_drug_effect['x_source'] = 'DrugBank'
df_drug_effect['y_type'] = 'effect/phenotype'
df_drug_effect['y_source'] = 'HPO'
df_drug_effect['relation'] = 'drug_effect'
df_drug_effect['display_relation'] = 'side effect'
df_drug_effect = df_drug_effect.query('y_id not in @hp_ids_r_mondo')
# df_drug_effect = add_reverse_edges(df_drug_effect)
df_drug_effect = clean_edges(df_drug_effect)
print(df_drug_effect.shape)
print("drug effect 关系数量:", df_drug_effect.shape[0])
print("drug effect 中 drug 数量:", df_drug_effect['x_id'].unique().shape[0])
print("drug effect 中 effect 数量:", df_drug_effect['y_id'].unique().shape[0])
print("drug effect 中 drug 的平均出度数:", df_drug_effect.groupby('x_id').size().mean())
print("drug effect 中 effect 的平均入度数:", df_drug_effect.groupby('y_id').size().mean())
df_drug_effect.head(1)

(60662, 10)
drug effect 关系数量: 60662
drug effect 中 drug 数量: 1114
drug effect 中 effect 数量: 959
drug effect 中 drug 的平均出度数: 54.45421903052065
drug effect 中 effect 的平均入度数: 63.25547445255474


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,drug_effect,side effect,DB00583,drug,Levocarnitine,DrugBank,2027,effect/phenotype,Abdominal pain,HPO


## GO Terms

### Go terms interactions (GO)

In [102]:
bp = df_go_terms.query('go_term_type=="biological_process"')
df_bp_bp = pd.merge(df_go_edges, bp, 'inner', left_on='x', right_on='go_term_id')
df_bp_bp = df_bp_bp.rename(columns={'go_term_id':'x_id','go_term_name':'x_name','go_term_type':'x_type'})
df_bp_bp = pd.merge(df_bp_bp, bp, 'inner', left_on='y', right_on='go_term_id')
df_bp_bp = df_bp_bp.rename(columns={'go_term_id':'y_id','go_term_name':'y_name','go_term_type':'y_type'})
df_bp_bp['relation'] = 'bioprocess_bioprocess'
df_bp_bp['x_source'] = 'GO'
df_bp_bp['y_source'] = 'GO'
df_bp_bp['display_relation'] = 'parent-child'
df_bp_bp = clean_edges(df_bp_bp)
print("biological process biological process 关系数量:", df_bp_bp.shape[0])
print("biological process biological process 中 bioprocess 数量:", pd.concat([df_bp_bp['x_id'], df_bp_bp['y_id']]).unique().shape[0])
print("biological process biological process 中 bioprocess 的平均出度数:", df_bp_bp.groupby('x_id').size().mean())
print("biological process biological process 中 bioprocess 的平均入度数:", df_bp_bp.groupby('y_id').size().mean())
df_bp_bp.head(1)

biological process biological process 关系数量: 45416
biological process biological process 中 bioprocess 数量: 26037
biological process biological process 中 bioprocess 的平均出度数: 3.74441421386759
biological process biological process 中 bioprocess 的平均入度数: 1.7443539714241818


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,bioprocess_bioprocess,parent-child,1903286,biological_process,regulation of potassium ion import,GO,1903288,biological_process,positive regulation of potassium ion import ac...,GO


In [104]:
mf = df_go_terms.query('go_term_type=="molecular_function"')
df_mf_mf = pd.merge(df_go_edges, mf, 'inner', left_on='x', right_on='go_term_id')
df_mf_mf = df_mf_mf.rename(columns={'go_term_id':'x_id','go_term_name':'x_name','go_term_type':'x_type'})
df_mf_mf = pd.merge(df_mf_mf, mf, 'inner', left_on='y', right_on='go_term_id')
df_mf_mf = df_mf_mf.rename(columns={'go_term_id':'y_id','go_term_name':'y_name','go_term_type':'y_type'})
df_mf_mf['relation'] = 'molfunc_molfunc'
df_mf_mf['display_relation'] = 'parent-child'
df_mf_mf['x_source'] = 'GO'
df_mf_mf['y_source'] = 'GO'
df_mf_mf = clean_edges(df_mf_mf)
print("molecular function molecular function 关系数量:", df_mf_mf.shape[0])
print("molecular function molecular function 中 molfunc 数量:", pd.concat([df_mf_mf['x_id'], df_mf_mf['y_id']]).unique().shape[0])
print("molecular function molecular function 中 molfunc 的平均出度数:", df_mf_mf.groupby('x_id').size().mean())
print("molecular function molecular function 中 molfunc 的平均入度数:", df_mf_mf.groupby('y_id').size().mean())
df_mf_mf.head(1)

molecular function molecular function 关系数量: 12572
molecular function molecular function 中 molfunc 数量: 10154
molecular function molecular function 中 molfunc 的平均出度数: 6.2954431647471205
molecular function molecular function 中 molfunc 的平均入度数: 1.2382547030434354


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,molfunc_molfunc,parent-child,1846,molecular_function,opsonin binding,GO,1851,molecular_function,complement component C3b binding,GO


In [105]:
cc = df_go_terms.query('go_term_type=="cellular_component"')
df_cc_cc = pd.merge(df_go_edges, cc, 'inner', left_on='x', right_on='go_term_id')
df_cc_cc = df_cc_cc.rename(columns={'go_term_id':'x_id','go_term_name':'x_name','go_term_type':'x_type'})
df_cc_cc = pd.merge(df_cc_cc, cc, 'inner', left_on='y', right_on='go_term_id')
df_cc_cc = df_cc_cc.rename(columns={'go_term_id':'y_id','go_term_name':'y_name','go_term_type':'y_type'})
df_cc_cc['relation'] = 'cellcomp_cellcomp'
df_cc_cc['display_relation'] = 'parent-child'
df_cc_cc['x_source'] = 'GO'
df_cc_cc['y_source'] = 'GO'
df_cc_cc = clean_edges(df_cc_cc)
print("cellular component cellular component 关系数量:", df_cc_cc.shape[0])
print("cellular component cellular component 中 cellcomp 数量:", pd.concat([df_cc_cc['x_id'], df_cc_cc['y_id']]).unique().shape[0])
print("cellular component cellular component 中 cellcomp 的平均出度数:", df_cc_cc.groupby('x_id').size().mean())
print("cellular component cellular component 中 cellcomp 的平均入度数:", df_cc_cc.groupby('y_id').size().mean())
df_cc_cc.head(1)


cellular component cellular component 关系数量: 4615
cellular component cellular component 中 cellcomp 数量: 4023
cellular component cellular component 中 cellcomp 的平均出度数: 5.416666666666667
cellular component cellular component 中 cellcomp 的平均入度数: 1.1474390850323222


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,cellcomp_cellcomp,parent-child,34707,cellular_component,chloride channel complex,GO,16935,cellular_component,glycine-gated chloride channel complex,GO


### Go protein interactions (Gene2GO)

In [106]:
df_prot_path = pd.merge(df_gene2go, df_go_terms, 'inner', 'go_term_id').rename(columns={'go_term_type_x':'go_term_type'})
df_prot_path = pd.merge(df_prot_path, df_prot_names, 'left', left_on='ncbi_gene_id', right_on='ncbi_id')
df_prot_path = df_prot_path.rename(columns={'ncbi_gene_id':'x_id', 'symbol':'x_name', 
                             'go_term_id':'y_id','go_term_name':'y_name', 'go_term_type':'y_type'})
df_prot_path['x_type'] = 'gene/protein'
df_prot_path['x_source'] = 'NCBI'
df_prot_path['y_source'] = 'GO'
df_prot_path = df_prot_path[['x_id','x_type', 'x_name', 'x_source','y_id','y_type', 'y_name', 'y_source']]

In [107]:
df_prot_mf = df_prot_path.query('y_type=="molecular_function"').copy()
df_prot_mf['relation'] = 'molfunc_protein'
df_prot_mf['display_relation'] = 'interacts with'
# df_prot_mf = add_reverse_edges(df_prot_mf)
df_prot_mf = clean_edges(df_prot_mf)
print("protein molecular function 关系数量:", df_prot_mf.shape[0])
print("protein molecular function 中 protein 数量:", df_prot_mf['x_id'].unique().shape[0])
print("protein molecular function 中 mf 数量:", df_prot_mf['y_id'].unique().shape[0])
print("protein molecular function 中 protein 的平均出度数:", df_prot_mf.groupby('x_id').size().mean())
print("protein molecular function 中 mf 的平均入度数:", df_prot_mf.groupby('y_id').size().mean())
df_prot_mf.head(1)

protein molecular function 关系数量: 92348
protein molecular function 中 protein 数量: 18582
protein molecular function 中 mf 数量: 4712
protein molecular function 中 protein 的平均出度数: 4.969755677537401
protein molecular function 中 mf 的平均入度数: 19.598471986417657


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,molfunc_protein,interacts with,2,gene/protein,A2M,NCBI,4867,molecular_function,serine-type endopeptidase inhibitor activity,GO


In [108]:
df_prot_cc = df_prot_path.query('y_type=="cellular_component"').copy()
df_prot_cc['relation'] = 'cellcomp_protein'
df_prot_cc['display_relation'] = 'interacts with'
# df_prot_cc = add_reverse_edges(df_prot_cc)
df_prot_cc = clean_edges(df_prot_cc)
print("protein cellular component 关系数量:", df_prot_cc.shape[0])
print("protein cellular component 中 protein 数量:", df_prot_cc['x_id'].unique().shape[0])
print("protein cellular component 中 cc 数量:", df_prot_cc['y_id'].unique().shape[0])
print("protein cellular component 中 protein 的平均出度数:", df_prot_cc.groupby('x_id').size().mean())
print("protein cellular component 中 cc 的平均入度数:", df_prot_cc.groupby('y_id').size().mean())
df_prot_cc.head(1)

protein cellular component 关系数量: 106485
protein cellular component 中 protein 数量: 19866
protein cellular component 中 cc 数量: 1848
protein cellular component 中 protein 的平均出度数: 5.3601630927212325
protein cellular component 中 cc 的平均入度数: 57.621753246753244


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
249894,cellcomp_protein,interacts with,1,gene/protein,A1BG,NCBI,5615,cellular_component,extracellular space,GO


In [109]:
df_prot_bp = df_prot_path.query('y_type=="biological_process"').copy()
df_prot_bp['relation'] = 'bioprocess_protein'
df_prot_bp['display_relation'] = 'interacts with'
# df_prot_bp = add_reverse_edges(df_prot_bp)
df_prot_bp = clean_edges(df_prot_bp)
print("protein biological process 关系数量:", df_prot_bp.shape[0])
print("protein biological process 中 protein 数量:", df_prot_bp['x_id'].unique().shape[0])
print("protein biological process 中 bp 数量:", df_prot_bp['y_id'].unique().shape[0])
print("protein biological process 中 protein 的平均出度数:", df_prot_bp.groupby('x_id').size().mean())
print("protein biological process 中 bp 的平均入度数:", df_prot_bp.groupby('y_id').size().mean())
df_prot_bp.head(1)

protein biological process 关系数量: 157420
protein biological process 中 protein 数量: 18758
protein biological process 中 bp 数量: 12245
protein biological process 中 protein 的平均出度数: 8.39215268152255
protein biological process 中 bp 的平均入度数: 12.85585953450388


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
92400,bioprocess_protein,interacts with,1,gene/protein,A1BG,NCBI,2764,biological_process,immune response-regulating signaling pathway,GO


## Exposure

### Exposure protein interactions (CTD)

In [110]:
df_exp_prot = df_exposures[['exposurestressorname', 'exposurestressorid','exposuremarker', 'exposuremarkerid']]
df_exp_prot = df_exp_prot.loc[df_exp_prot[['exposuremarkerid']].dropna().index, :]

# 只使用了 id 为数字的部分 (NCBI 的 protein)
gene_row_index = []
for idx, data in df_exp_prot.iterrows():
    if data.exposuremarkerid.isnumeric(): 
        gene_row_index.append(idx)

df_exp_prot = df_exp_prot.loc[gene_row_index, :].astype({'exposuremarkerid': 'int'}).astype({'exposuremarkerid': 'str'})

df_exp_prot = pd.merge(df_exp_prot, df_prot_names, 'left', left_on='exposuremarkerid', right_on='ncbi_id')

df_exp_prot = df_exp_prot.rename(columns={'exposurestressorid':'x_id', 'exposurestressorname':'x_name', 'ncbi_id':'y_id', 'symbol':'y_name'})
df_exp_prot['x_type'] = 'exposure'
df_exp_prot['x_source'] = 'CTD'
df_exp_prot['y_type'] = 'gene/protein'
df_exp_prot['y_source'] = 'NCBI'
df_exp_prot['relation'] = 'exposure_protein'
df_exp_prot['display_relation'] = 'interacts with'
# df_exp_prot = add_reverse_edges(df_exp_prot)
df_exp_prot = clean_edges(df_exp_prot)
print("exposure protein 关系数量:", df_exp_prot.shape[0])
print("exposure protein 中 exposure 数量:", df_exp_prot['x_id'].unique().shape[0])
print("exposure protein 中 protein 数量:", df_exp_prot['y_id'].unique().shape[0])
print("exposure protein 中 exposure 的平均出度数:", df_exp_prot.groupby('x_id').size().mean())
print("exposure protein 中 protein 的平均入度数:", df_exp_prot.groupby('y_id').size().mean())
df_exp_prot.head(1)

exposure protein 关系数量: 2941
exposure protein 中 exposure 数量: 217
exposure protein 中 protein 数量: 1503
exposure protein 中 exposure 的平均出度数: 13.55299539170507
exposure protein 中 protein 的平均入度数: 1.9567531603459747


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,exposure_protein,interacts with,C092102,exposure,1-hydroxyphenanthrene,CTD,1401,gene/protein,CRP,NCBI


### Exposure disease interactions (CTD)

In [111]:
df_exp_dis = df_exposures[['exposurestressorname', 'exposurestressorid','diseasename', 'diseaseid']]
df_exp_dis = df_exp_dis.loc[df_exp_dis[['diseaseid']].dropna().index, :]
df_exp_dis = pd.merge(df_exp_dis, df_mondo_xref.query('ontology=="MESH"'), 'left', left_on='diseaseid', right_on='ontology_id')  # 使用 MeSH 联系 CTD
df_exp_dis = pd.merge(df_exp_dis, df_mondo_terms, 'left', left_on='mondo_id', right_on= 'id')

df_exp_dis = df_exp_dis.rename(columns={'exposurestressorid':'x_id', 'exposurestressorname':'x_name', 'mondo_id':'y_id', 'name':'y_name'})
df_exp_dis['x_type'] = 'exposure'
df_exp_dis['x_source'] = 'CTD'
df_exp_dis['y_type'] = 'disease'
df_exp_dis['y_source'] = 'MONDO'
df_exp_dis['relation'] = 'exposure_disease'
df_exp_dis['display_relation'] = 'linked to'
# df_exp_dis = add_reverse_edges(df_exp_dis)
df_exp_dis = clean_edges(df_exp_dis)
print("exposure disease 关系数量:", df_exp_dis.shape[0])
print("exposure disease 中 exposure 数量:", df_exp_dis['x_id'].unique().shape[0])
print("exposure disease 中 disease 数量:", df_exp_dis['y_id'].unique().shape[0])
print("exposure disease 中 exposure 的平均出度数:", df_exp_dis.groupby('x_id').size().mean())
print("exposure disease 中 disease 的平均入度数:", df_exp_dis.groupby('y_id').size().mean())
df_exp_dis.head(1)

exposure disease 关系数量: 2416
exposure disease 中 exposure 数量: 472
exposure disease 中 disease 数量: 326
exposure disease 中 exposure 的平均出度数: 5.11864406779661
exposure disease 中 disease 的平均入度数: 7.411042944785276


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,exposure_disease,linked to,C024566,exposure,"1,1,1-trichloroethane",CTD,4976,disease,amyotrophic lateral sclerosis,MONDO


### Exposure exposure interactions (CTD)

In [112]:
exposures = df_exposures['exposurestressorid'].unique().tolist()  # 获取所有 exposurestressorid
df_exp_exp = df_exposures.query('exposuremarkerid in @exposures')  # marker 同样为 exposures

df_exp_exp = df_exp_exp[['exposurestressorname', 'exposurestressorid','exposuremarker', 'exposuremarkerid']]
df_exp_exp = df_exp_exp.loc[df_exp_exp[['exposuremarkerid']].dropna().index, :]
df_exp_exp = df_exp_exp.drop_duplicates()

df_exp_exp = df_exp_exp.rename(columns={'exposurestressorid':'x_id', 'exposurestressorname':'x_name', 'exposuremarker':'y_name', 'exposuremarkerid':'y_id'})
df_exp_exp['x_type'] = 'exposure'
df_exp_exp['x_source'] = 'CTD'
df_exp_exp['y_type'] = 'exposure'
df_exp_exp['y_source'] = 'CTD'
df_exp_exp['relation'] = 'exposure_exposure'
df_exp_exp['display_relation'] = 'parent-child'
df_exp_exp = clean_edges(df_exp_exp)
print("exposure exposure 关系数量:", df_exp_exp.shape[0])
print("exposure exposure 中 exposure 数量:", pd.concat([df_exp_exp['x_id'], df_exp_exp['y_id']]).unique().shape[0])
print("exposure exposure 中 exposure 的平均出度数:", df_exp_exp.groupby('x_id').size().mean())
print("exposure exposure 中 exposure 的平均入度数:", df_exp_exp.groupby('y_id').size().mean())
print(df_exp_exp.shape)
df_exp_exp.head(1)

exposure exposure 关系数量: 2505
exposure exposure 中 exposure 数量: 698
exposure exposure 中 exposure 的平均出度数: 8.21311475409836
exposure exposure 中 exposure 的平均入度数: 4.154228855721393
(2505, 10)


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
845,exposure_exposure,parent-child,C029350,exposure,1-naphthol,CTD,D004958,exposure,Estradiol,CTD


### Exposure pathway interactions (CTD)

In [114]:
# phenotypes are actually pathways 

df_exp_path = df_exposures[['exposurestressorname', 'exposurestressorid','phenotypename', 'phenotypeid']]
df_exp_path = df_exp_path.loc[df_exp_path[['phenotypeid']].dropna().index, :]

df_exp_path.loc[:, 'phenotypeid'] = [str(int(x.split(':')[1])) for x in df_exp_path[['phenotypeid']].values.reshape(-1)]  # 只有 GO
df_exp_path = df_exp_path.drop_duplicates()
df_exp_path = pd.merge(df_exp_path, df_go_terms, 'inner', left_on='phenotypeid', right_on='go_term_id')
df_exp_path = df_exp_path.rename(columns={'exposurestressorid':'x_id', 'exposurestressorname':'x_name', 
                                          'go_term_id':'y_id', 'go_term_name':'y_name', 'go_term_type':'y_type'})
df_exp_path['x_type'] = 'exposure'
df_exp_path['x_source'] = 'CTD'
df_exp_path['y_source'] = 'GO'

In [115]:
df_exp_bp = df_exp_path.query('y_type=="biological_process"').copy()
df_exp_bp['relation'] = 'exposure_bioprocess'
df_exp_bp['display_relation'] = 'interacts with'
# df_exp_bp = add_reverse_edges(df_exp_bp)
df_exp_bp = clean_edges(df_exp_bp)
print(df_exp_bp.shape)
print("exposure biological process 关系数量:", df_exp_bp.shape[0])
print("exposure biological process 中 exposure 数量:", df_exp_bp['x_id'].unique().shape[0])
print("exposure biological process 中 bp 数量:", df_exp_bp['y_id'].unique().shape[0])
print("exposure biological process 中 exposure 的平均出度数:", df_exp_bp.groupby('x_id').size().mean())
print("exposure biological process 中 bp 的平均入度数:", df_exp_bp.groupby('y_id').size().mean())
df_exp_bp.head(1)

(2092, 10)
exposure biological process 关系数量: 2092
exposure biological process 中 exposure 数量: 368
exposure biological process 中 bp 数量: 460
exposure biological process 中 exposure 的平均出度数: 5.684782608695652
exposure biological process 中 bp 的平均入度数: 4.547826086956522


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,exposure_bioprocess,interacts with,C006718,exposure,"1,12-benzoperylene",CTD,42632,biological_process,cholesterol homeostasis,GO


In [116]:
df_exp_mf = df_exp_path.query('y_type=="molecular_function"').copy()
df_exp_mf['relation'] = 'exposure_molfunc'
df_exp_mf['display_relation'] = 'interacts with'
# df_exp_mf = add_reverse_edges(df_exp_mf)
df_exp_mf = clean_edges(df_exp_mf)
print("exposure molecular function 关系数量:", df_exp_mf.shape[0])
print("exposure molecular function 中 exposure 数量:", df_exp_mf['x_id'].unique().shape[0])
print("exposure molecular function 中 mf 数量:", df_exp_mf['y_id'].unique().shape[0])
print("exposure molecular function 中 exposure 的平均出度数:", df_exp_mf.groupby('x_id').size().mean())
print("exposure molecular function 中 mf 的平均入度数:", df_exp_mf.groupby('y_id').size().mean())
df_exp_mf.head(1)

exposure molecular function 关系数量: 47
exposure molecular function 中 exposure 数量: 31
exposure molecular function 中 mf 数量: 18
exposure molecular function 中 exposure 的平均出度数: 1.5161290322580645
exposure molecular function 中 mf 的平均入度数: 2.611111111111111


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
94,exposure_molfunc,interacts with,C014024,exposure,"2,4,5,2',4',5'-hexachlorobiphenyl",CTD,19766,molecular_function,IgA receptor activity,GO


In [117]:
df_exp_cc = df_exp_path.query('y_type=="cellular_component"').copy()
df_exp_cc['relation'] = 'exposure_cellcomp'
df_exp_cc['display_relation'] = 'interacts with'
# df_exp_cc = add_reverse_edges(df_exp_cc)
df_exp_cc = clean_edges(df_exp_cc)
print("exposure cellular component 关系数量:", df_exp_cc.shape[0])
print("exposure cellular component 中 exposure 数量:", df_exp_cc['x_id'].unique().shape[0])
print("exposure cellular component 中 cc 数量:", df_exp_cc['y_id'].unique().shape[0])
print("exposure cellular component 中 exposure 的平均出度数:", df_exp_cc.groupby('x_id').size().mean())
print("exposure cellular component 中 cc 的平均入度数:", df_exp_cc.groupby('y_id').size().mean())
df_exp_cc.head(1)

exposure cellular component 关系数量: 13
exposure cellular component 中 exposure 数量: 12
exposure cellular component 中 cc 数量: 4
exposure cellular component 中 exposure 的平均出度数: 1.0833333333333333
exposure cellular component 中 cc 的平均入度数: 3.25


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
228,exposure_cellcomp,interacts with,D000393,exposure,Air Pollutants,CTD,71743,cellular_component,"IgE immunoglobulin complex, circulating",GO


## Anatomy

### Anatomy anatomy interactions (UBERON) 

In [118]:
df_ana_ana = pd.merge(df_uberon_is_a, df_uberon_terms, 'left', left_on='id', right_on='id')
df_ana_ana = df_ana_ana.rename(columns={'id':'x_id', 'name':'x_name'})
df_ana_ana = pd.merge(df_ana_ana, df_uberon_terms, 'left', left_on='is_a', right_on='id')
df_ana_ana = df_ana_ana.rename(columns={'id':'y_id', 'name':'y_name'})
df_ana_ana['x_type'] = 'anatomy'
df_ana_ana['x_source'] = 'UBERON'
df_ana_ana['y_type'] = 'anatomy'
df_ana_ana['y_source'] = 'UBERON'
df_ana_ana['relation'] = 'anatomy_anatomy'
df_ana_ana['display_relation'] = 'parent-child'
df_ana_ana = clean_edges(df_ana_ana)
print("anatomy anatomy 关系数量:", df_ana_ana.shape[0])
print("anatomy anatomy 中 anatomy 数量:", pd.concat([df_ana_ana['x_id'], df_ana_ana['y_id']]).unique().shape[0])
print("anatomy anatomy 中 anatomy 的平均出度数:", df_ana_ana.groupby('x_id').size().mean())
print("anatomy anatomy 中 anatomy 的平均入度数:", df_ana_ana.groupby('y_id').size().mean())
df_ana_ana.head(1)

anatomy anatomy 关系数量: 14595
anatomy anatomy 中 anatomy 数量: 14597
anatomy anatomy 中 anatomy 的平均出度数: 1.0
anatomy anatomy 中 anatomy 的平均入度数: 4.782110091743119


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,anatomy_anatomy,parent-child,2,anatomy,uterine cervix,UBERON,5156,anatomy,reproductive structure,UBERON


### Anatomy Protein (BGEE)

In [119]:
df_bgee = pd.merge(df_bgee, df_prot_names, 'inner', left_on='gene_name', right_on='symbol')

df_bgee = df_bgee.rename(columns={'ncbi_id':'x_id', 'symbol':'x_name', 
                                  'anatomy_id':'y_id', 'anatomy_name':'y_name'})
df_bgee['x_source'] = 'NCBI'
df_bgee['x_type'] = 'gene/protein'
df_bgee['y_source'] = 'UBERON'
df_bgee['y_type'] = 'anatomy'

In [120]:
df_ana_prot_pos = df_bgee.query('expression=="present"').copy()
df_ana_prot_pos['relation'] = 'anatomy_protein_present'
df_ana_prot_pos['display_relation'] = 'expression present'
# df_ana_prot_pos = add_reverse_edges(df_ana_prot_pos)
df_ana_prot_pos = clean_edges(df_ana_prot_pos)
print("anatomy protein present 关系数量:", df_ana_prot_pos.shape[0])
print("anatomy protein present 中 anatomy 数量:", df_ana_prot_pos['x_id'].unique().shape[0])
print("anatomy protein present 中 protein 数量:", df_ana_prot_pos['y_id'].unique().shape[0])
print("anatomy protein present 中 anatomy 的平均出度数:", df_ana_prot_pos.groupby('x_id').size().mean())
print("anatomy protein present 中 protein 的平均入度数:", df_ana_prot_pos.groupby('y_id').size().mean())
df_ana_prot_pos.head(1)

anatomy protein present 关系数量: 3816181
anatomy protein present 中 anatomy 数量: 33605
anatomy protein present 中 protein 数量: 310
anatomy protein present 中 anatomy 的平均出度数: 113.55991667906561
anatomy protein present 中 protein 的平均入度数: 12310.26129032258


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,anatomy_protein_present,expression present,7105,gene/protein,TSPAN6,NCBI,2,anatomy,uterine cervix,UBERON


In [121]:
df_ana_prot_neg = df_bgee.query('expression=="absent"').copy()
df_ana_prot_neg['relation'] = 'anatomy_protein_absent'
df_ana_prot_neg['display_relation'] = 'expression absent'
# df_ana_prot_neg = add_reverse_edges(df_ana_prot_neg)
df_ana_prot_neg = clean_edges(df_ana_prot_neg)
print("anatomy protein absent 关系数量:", df_ana_prot_neg.shape[0])
print("anatomy protein absent 中 anatomy 数量:", df_ana_prot_neg['x_id'].unique().shape[0])
print("anatomy protein absent 中 protein 数量:", df_ana_prot_neg['y_id'].unique().shape[0])
print("anatomy protein absent 中 anatomy 的平均出度数:", df_ana_prot_neg.groupby('x_id').size().mean())
print("anatomy protein absent 中 protein 的平均入度数:", df_ana_prot_neg.groupby('y_id').size().mean())
df_ana_prot_neg.head(1)

anatomy protein absent 关系数量: 368272
anatomy protein absent 中 anatomy 数量: 16787
anatomy protein absent 中 protein 数量: 207
anatomy protein absent 中 anatomy 的平均出度数: 21.937928158694227
anatomy protein absent 中 protein 的平均入度数: 1779.0917874396134


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
49,anatomy_protein_absent,expression absent,7105,gene/protein,TSPAN6,NCBI,1103,anatomy,diaphragm,UBERON


## Pathways

In [122]:
df_path_path = pd.merge(df_reactome_rels, df_reactome_terms, 'inner', left_on='reactome_id_1', right_on='reactome_id')
df_path_path = df_path_path.rename(columns={'reactome_id': 'x_id', 'reactome_name':'x_name'})
df_path_path = pd.merge(df_path_path, df_reactome_terms, 'inner', left_on='reactome_id_2', right_on='reactome_id')
df_path_path = df_path_path.rename(columns={'reactome_id': 'y_id', 'reactome_name':'y_name'})

df_path_path['x_source'] = 'REACTOME'
df_path_path['x_type'] = 'pathway'
df_path_path['y_source'] = 'REACTOME'
df_path_path['y_type'] = 'pathway'
df_path_path['relation'] = 'pathway_pathway'
df_path_path['display_relation'] = 'parent-child'
df_path_path = clean_edges(df_path_path)
print("pathway pathway 关系数量:", df_path_path.shape[0])
print("pathway pathway 中 pathway 数量:", pd.concat([df_path_path['x_id'], df_path_path['y_id']]).unique().shape[0])
print("pathway pathway 中 pathway 的平均出度数:", df_path_path.groupby('x_id').size().mean())
print("pathway pathway 中 pathway 的平均入度数:", df_path_path.groupby('y_id').size().mean())
df_path_path.head(1)

pathway pathway 关系数量: 2787
pathway pathway 中 pathway 数量: 2769
pathway pathway 中 pathway 的平均出度数: 3.4407407407407407
pathway pathway 中 pathway 的平均入度数: 1.0171532846715328


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,pathway_pathway,parent-child,R-HSA-109581,pathway,Apoptosis,REACTOME,R-HSA-109606,pathway,Intrinsic Pathway for Apoptosis,REACTOME


### Pathway protein interactions

In [123]:
df_path_prot = pd.merge(df_reactome_ncbi, df_prot_names, 'inner', 'ncbi_id')

df_path_prot = df_path_prot.rename(columns={'ncbi_id': 'x_id', 'symbol':'x_name', 
                                            'reactome_id': 'y_id', 'reactome_name':'y_name'})
df_path_prot['x_source'] = 'NCBI'
df_path_prot['x_type'] = 'gene/protein'
df_path_prot['y_source'] = 'REACTOME'
df_path_prot['y_type'] = 'pathway'
df_path_prot['relation'] = 'pathway_protein'
df_path_prot['display_relation'] = 'interacts with'
# df_path_prot = add_reverse_edges(df_path_prot)
df_path_prot = clean_edges(df_path_prot)
print("pathway protein 关系数量:", df_path_prot.shape[0])
print("pathway protein 中 pathway 数量:", df_path_prot['x_id'].unique().shape[0])
print("pathway protein 中 protein 数量:", df_path_prot['y_id'].unique().shape[0])
print("pathway protein 中 pathway 的平均出度数:", df_path_prot.groupby('x_id').size().mean())
print("pathway protein 中 protein 的平均入度数:", df_path_prot.groupby('y_id').size().mean())
df_path_prot.head(1)

pathway protein 关系数量: 45665
pathway protein 中 pathway 数量: 11205
pathway protein 中 protein 数量: 2218
pathway protein 中 pathway 的平均出度数: 4.0754127621597505
pathway protein 中 protein 的平均入度数: 20.588367899008116


,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,pathway_protein,interacts with,1,gene/protein,A1BG,NCBI,R-HSA-114608,pathway,Platelet degranulation,REACTOME


# Compiling knowledge graph

In [ ]:
# kg = pd.concat([df_prot_prot, df_prot_drug, df_drug_dis, df_drug_drug, df_prot_phe,
#                 df_phe_phe, df_dis_phe_neg, df_dis_phe_pos, df_prot_dis, df_dis_dis, 
#                 df_drug_effect, df_bp_bp, df_mf_mf, df_cc_cc, df_prot_mf, 
#                 df_prot_cc, df_prot_bp, df_exp_prot, df_exp_dis, df_exp_exp, 
#                 df_exp_bp, df_exp_mf, df_exp_cc, df_path_path, df_path_prot,
#                 df_ana_ana, df_ana_prot_pos, df_ana_prot_neg, df_drug_path, df_path_path_add], ignore_index=True) #28 -> 30
# kg = clean_edges(kg)
# kg.tail()

In [124]:
kg = pd.concat([df_prot_prot, df_prot_drug, df_drug_dis, df_drug_drug, df_prot_phe,
                df_phe_phe, df_dis_phe_neg, df_dis_phe_pos, df_prot_dis, df_dis_dis, 
                df_drug_effect, df_bp_bp, df_mf_mf, df_cc_cc, df_prot_mf, 
                df_prot_cc, df_prot_bp, df_path_path, df_path_prot, df_drug_path, df_path_path_add], ignore_index=True)  # 21
kg = clean_edges(kg)
kg.head()

,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,protein_protein,ppi,381,gene/protein,ARF5,NCBI,4907,gene/protein,NT5E,NCBI
1,protein_protein,ppi,381,gene/protein,ARF5,NCBI,1845,gene/protein,DUSP3,NCBI
2,protein_protein,ppi,381,gene/protein,ARF5,NCBI,84364,gene/protein,ARFGAP2,NCBI
3,protein_protein,ppi,381,gene/protein,ARF5,NCBI,23071,gene/protein,ERP44,NCBI
4,protein_protein,ppi,381,gene/protein,ARF5,NCBI,10972,gene/protein,TMED10,NCBI


In [125]:
kg['x_id'] = kg['x_id'].apply(lambda x: 'kg4rd:' + x)
kg['y_id'] = kg['y_id'].apply(lambda x: 'kg4rd:' + x)
kg.head()

,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:4907,gene/protein,NT5E,NCBI
1,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:1845,gene/protein,DUSP3,NCBI
2,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:84364,gene/protein,ARFGAP2,NCBI
3,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:23071,gene/protein,ERP44,NCBI
4,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:10972,gene/protein,TMED10,NCBI


In [126]:
kg.to_csv(save_path+'auxiliary/kg_raw.csv', index=False)

# Get giant component

In [127]:
kg = pd.read_csv(save_path+'auxiliary/kg_raw.csv', low_memory=False)
kg.head()

,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:4907,gene/protein,NT5E,NCBI
1,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:1845,gene/protein,DUSP3,NCBI
2,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:84364,gene/protein,ARFGAP2,NCBI
3,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:23071,gene/protein,ERP44,NCBI
4,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:10972,gene/protein,TMED10,NCBI


In [128]:
nodes = pd.concat([kg[['x_id','x_type', 'x_name','x_source']].rename(columns={'x_id':'node_id', 'x_type':'node_type', 'x_name':'node_name','x_source':'node_source'}), 
                   kg[['y_id','y_type', 'y_name','y_source']].rename(columns={'y_id':'node_id', 'y_type':'node_type', 'y_name':'node_name','y_source':'node_source'})])

nodes = nodes.drop_duplicates().reset_index().drop('index',axis=1).reset_index().rename(columns={'index':'node_idx'})

edges = pd.merge(kg, nodes, 'left', left_on=['x_id','x_type', 'x_name','x_source'], right_on=['node_id','node_type','node_name','node_source'])
edges = edges.rename(columns={'node_idx':'x_idx'})
edges = pd.merge(edges, nodes, 'left', left_on=['y_id','y_type', 'y_name','y_source'], right_on=['node_id','node_type','node_name','node_source'])
edges = edges.rename(columns={'node_idx':'y_idx'})

edges = edges[['relation', 'display_relation','x_idx', 'y_idx']]
edges['combine_idx'] = edges['x_idx'].astype(str) + '-' + edges['y_idx'].astype(str)

edge_index = edges[['x_idx', 'y_idx']].values.T

graph = ig.Graph()
graph.add_vertices(list(range(nodes.shape[0])))
graph.add_edges([tuple(x) for x in edge_index.T])

graph = graph.as_undirected(mode='collapse')

c = graph.components(mode='strong')
giant = c.giant()

print('Nodes: %d' % giant.vcount())
print('Edges: %d' % giant.ecount())

assert not giant.is_directed()
assert giant.is_connected()

giant_nodes = giant.vs['name']
new_nodes = nodes.query('node_idx in @giant_nodes')
assert new_nodes.shape[0] == giant.vcount()

new_edges = edges.query('x_idx in @giant_nodes and y_idx in @giant_nodes').copy()
assert new_edges.shape[0] == giant.ecount()

new_kg = pd.merge(new_edges, new_nodes, 'left', left_on='x_idx', right_on='node_idx')
new_kg = new_kg.rename(columns={'node_id':'x_id', 'node_type':'x_type', 'node_name':'x_name','node_source':'x_source'}) 
new_kg = pd.merge(new_kg, new_nodes, 'left', left_on='y_idx', right_on='node_idx')
new_kg = new_kg.rename(columns={'node_id':'y_id', 'node_type':'y_type', 'node_name':'y_name','node_source':'y_source'}) 
new_kg = clean_edges(new_kg)

Nodes: 121649
Edges: 4617425


In [129]:
kg = new_kg.copy()
kg.to_csv(save_path+'auxiliary/kg_giant.csv', index=False)

In [130]:
kg = pd.read_csv(save_path+'auxiliary/kg_giant.csv', low_memory=False)
kg.head()

,relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
0,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:4907,gene/protein,NT5E,NCBI
1,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:1845,gene/protein,DUSP3,NCBI
2,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:84364,gene/protein,ARFGAP2,NCBI
3,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:23071,gene/protein,ERP44,NCBI
4,protein_protein,ppi,kg4rd:381,gene/protein,ARF5,NCBI,kg4rd:10972,gene/protein,TMED10,NCBI


In [131]:
nodes = pd.concat([kg[['x_id','x_type', 'x_name','x_source']].rename(columns={'x_id':'node_id', 'x_type':'node_type', 'x_name':'node_name', 'x_source':'node_source'}), 
                   kg[['y_id','y_type', 'y_name','y_source']].rename(columns={'y_id':'node_id', 'y_type':'node_type', 'y_name':'node_name', 'y_source':'node_source'})])
nodes = nodes.drop_duplicates().reset_index().drop('index',axis=1).reset_index().rename(columns={'index':'node_index'})

kg = pd.merge(kg, nodes.rename(columns={'node_index':'x_index',
                                        'node_id':'x_id',
                                        'node_type':'x_type',
                                        'node_name':'x_name',
                                        'node_source':'x_source'}), 'left').dropna()
kg = pd.merge(kg, nodes.rename(columns={'node_index':'y_index',
                                        'node_id':'y_id',
                                        'node_type':'y_type',
                                        'node_name':'y_name',
                                        'node_source':'y_source'}), 'left').dropna()
kg = kg[['relation', 'display_relation', 'x_index', 'x_id', 'x_type', 'x_name', 'x_source', 'y_index', 'y_id', 'y_type', 'y_name', 'y_source']]

edges = kg[['relation', 'display_relation', 'x_index', 'y_index']].copy()

In [132]:
kg.to_csv(save_path+'kg.csv', index=False)
nodes.to_csv(save_path+'nodes.csv', index=False)
edges.to_csv(save_path+'edges.csv', index=False)